# Refresh specific partitions of a semantic model

Uses [`fabric-partition-refresh`](https://github.com/akg-byte/fabric-partition-refresh). Run top to bottom. **Nothing is refreshed until you set `DRY_RUN = False`.**

In [ ]:
%pip install fabric-partition-refresh

In [ ]:
import logging

from fabric_partition_refresh import (
    PartitionSelector,
    RefreshOptions,
    SempyBackend,
    list_partitions,
    refresh_partitions,
    summarize_tables,
)

logging.basicConfig(level=logging.INFO, format="%(message)s")

## 1. Connect
Only the workspace and semantic model are needed up front.

In [ ]:
WORKSPACE = "My Workspace"
SEMANTIC_MODEL = "My Semantic Model"

backend = SempyBackend(WORKSPACE, SEMANTIC_MODEL)

## 2. Discover tables
Find the exact table name you want to refresh.

In [ ]:
display(summarize_tables(backend))

## 3. Browse the partitions of one table
Set `TABLE` from the list above, then run to see its partition names.

In [ ]:
TABLE = "FactSales"

display(list_partitions(backend, TABLE))

## 4. Choose what to refresh
Using the partition names above. Filters combine with AND; leave all commented to select every partition of the table.

In [ ]:
# Safety switch: True = plan only.
DRY_RUN = True

selector = PartitionSelector(
    table=TABLE,
    # contains="2025",
    # pattern=r"^2025Q[12]",
    # names=("2025Q101", "2025Q102"),
    # min_partition="2024Q410",
    # max_partition="2025Q412",
)

options = RefreshOptions(
    dry_run=DRY_RUN,
    strategy="auto",  # auto | enhanced | tmsl
    refresh_type="full",
    batch_size=None,  # e.g. 4 = refresh 4 partitions per request
    retry_count=2,
)

## 5. Plan / refresh

In [ ]:
report = refresh_partitions(WORKSPACE, SEMANTIC_MODEL, selector, options, backend=backend)
print("Status:", report.status)
display(report.selected)

## 6. Verify and audit (after a real run)

In [ ]:
if report.verification is not None:
    display(report.verification)
    display(report.to_audit_frame())